In [13]:
!pip install tensorflow pandas numpy scikit-learn matplotlib nltk

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
import re

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from tensorflow.keras.callbacks import EarlyStopping

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
import zipfile

zip_path = "/content/archive.zip"   # Change this if your zip has a different name

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("Dataset Extracted Successfully!")

Dataset Extracted Successfully!


In [16]:
import os

print(os.listdir('/content/dataset'))

['True.csv', 'Fake.csv']


In [17]:
fake = pd.read_csv('/content/dataset/Fake.csv')
true = pd.read_csv('/content/dataset/True.csv')

print(fake.head())
print(true.head())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept t

In [18]:
print("Fake News:", fake.shape)
print("True News:", true.shape)

Fake News: (23481, 4)
True News: (21417, 4)


In [19]:
fake["label"] = 0   # Fake News
true["label"] = 1   # Real News

In [20]:
data = pd.concat([fake, true], axis=0)

print(data.shape)
data.head()

(44898, 5)


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [21]:
data = data.sample(frac=1, random_state=42)
data.reset_index(drop=True, inplace=True)

data.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [22]:
data = data[['title', 'text', 'label']]

In [23]:
data.head()

,title,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1


In [24]:
print(data.isnull().sum())

title    0
text     0
label    0
dtype: int64


In [25]:
data.dropna(inplace=True)

In [26]:
data["content"] = data["title"] + " " + data["text"]

data = data[["content", "label"]]

data.head()

,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,1


In [27]:
from nltk.corpus import stopwords
import re

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()                     # Convert to lowercase
    text = re.sub(r"http\S+", "", text)     # Remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # Keep only letters
    text = text.split()                     # Split into words
    text = [word for word in text if word not in stop_words]
    text = " ".join(text)                   # Join back into a sentence
    return text

In [28]:
data["content"] = data["content"].apply(clean_text)

data.head()

,content,label
0,ben stein calls th circuit court committed cou...,0
1,trump drops steve bannon national security cou...,1
2,puerto rico expects u lift jones act shipping ...,1
3,oops trump accidentally confirmed leaked israe...,0
4,donald trump heads scotland reopen golf resort...,1


In [29]:
X = data["content"]
y = data["label"]

In [30]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X)

X = tokenizer.texts_to_sequences(X)

In [31]:
X = pad_sequences(X, maxlen=300)

print(X.shape)

(44898, 300)


In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(35918, 300)
(8980, 300)


In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [38]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 460s 1s/step - accuracy: 0.9853 - loss: 0.0451 - val_accuracy: 0.9882 - val_loss: 0.0380
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 461s 1s/step - accuracy: 0.9846 - loss: 0.0449 - val_accuracy: 0.9843 - val_loss: 0.0481
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 460s 1s/step - accuracy: 0.9820 - loss: 0.0593 - val_accuracy: 0.9833 - val_loss: 0.0559
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 462s 1s/step - accuracy: 0.9908 - loss: 0.0279 - val_accuracy: 0.9854 - val_loss: 0.0520
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 459s 1s/step - accuracy: 0.9942 - loss: 0.0185 - val_accuracy: 0.9918 - val_loss: 0.0315


In [39]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

281/281 ━━━━━━━━━━━━━━━━━━━━ 59s 211ms/step - accuracy: 0.9891 - loss: 0.0420
Test Accuracy: 0.9890868663787842


In [40]:
model.save("fake_news_model.keras")

In [41]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [42]:
from google.colab import files

files.download("fake_news_model.keras")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [44]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 472s 1s/step - accuracy: 0.9966 - loss: 0.0111 - val_accuracy: 0.9813 - val_loss: 0.0663
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 463s 1s/step - accuracy: 0.9975 - loss: 0.0087 - val_accuracy: 0.9889 - val_loss: 0.0534
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 506s 1s/step - accuracy: 0.9984 - loss: 0.0062 - val_accuracy: 0.9882 - val_loss: 0.0505
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 484s 1s/step - accuracy: 0.9977 - loss: 0.0075 - val_accuracy: 0.9910 - val_loss: 0.0405
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 460s 1s/step - accuracy: 0.9997 - loss: 0.0012 - val_accuracy: 0.9896 - val_loss: 0.0591


In [45]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

281/281 ━━━━━━━━━━━━━━━━━━━━ 53s 188ms/step - accuracy: 0.9894 - loss: 0.0573
Test Accuracy: 0.9894209504127502


In [46]:
model.save("fake_news_model.keras")

In [47]:
import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

In [48]:
def predict_news(news):
    news = clean_text(news)

    seq = tokenizer.texts_to_sequences([news])
    padded = pad_sequences(seq, maxlen=300)

    prediction = model.predict(padded)

    if prediction[0][0] > 0.5:
        print("✅ Real News")
        print("Confidence:", round(prediction[0][0]*100,2),"%")
    else:
        print("❌ Fake News")
        print("Confidence:", round((1-prediction[0][0])*100,2),"%")

In [49]:
predict_news("NASA discovers water on Mars")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 540ms/step
❌ Fake News
Confidence: 96.53 %


In [50]:
predict_news("Aliens have taken over the White House")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step
❌ Fake News
Confidence: 100.0 %


In [51]:
predict_news("Government announces new education policy")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
❌ Fake News
Confidence: 100.0 %
